# putEMG — Multi-Model Training & Evaluation

Trains and evaluates 5 architectures on the putEMG gesture dataset using a
single train/val split with early stopping and learning-rate scheduling,
then evaluates each model on the held-out test set.

| # | Model | Description |
|---|-------|-------------|
| 1 | **EEGNet** | Depthwise separable CNN, original baseline |
| 2 | **ShallowConvNet** | Temporal + spatial conv, square/log nonlinearity |
| 3 | **DeepConvNet** | 4 stacked conv blocks, increasing filter depth |
| 4 | **CNN_LSTM** | Spatial CNN → temporal pooling → 2-layer LSTM |
| 5 | **EMG_TCN** | Spatial mixing + 4 dilated temporal conv blocks |

**Data splits:**
- 80% training set (further split 90/10 into train/val for early stopping)
- 20% held-out test set → final evaluation only

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau

import models
from data_utils import train, evaluate, evaluateFinal
from data_utils import train_loader, dev_loader, test_loader

In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


---
## Configuration

Set paths, CV parameters, and hyperparameter grid here.

In [ ]:
# ── Training ──────────────────────────────────────────────────────────────────
MAX_EPOCHS   = 50
PATIENCE     = 15       # early-stopping patience (epochs without improvement)
MIN_DELTA    = 0.002    # minimum improvement to reset patience

# ── Hyperparameters ───────────────────────────────────────────────────────────
DROPOUT      = 0.25
LR           = 1e-3

---
## Training

Each model is trained on the training set with:
- **Early stopping** — stops when val accuracy doesn't improve by `MIN_DELTA` for `PATIENCE` epochs
- **ReduceLROnPlateau** — halves the LR when val accuracy plateaus for 5 epochs
- **Best-state checkpointing** — restores the best-val-acc weights before test evaluation

In [ ]:
model_registry = {
    'EEGNet':         lambda: models.EEGNet(dropout_rate=DROPOUT),
    'ShallowConvNet': lambda: models.ShallowConvNet(dropout_rate=DROPOUT),
    'DeepConvNet':    lambda: models.DeepConvNet(dropout_rate=DROPOUT),
    'CNN_LSTM':       lambda: models.CNN_LSTM(dropout_rate=DROPOUT),
    'EMG_TCN':        lambda: models.EMG_TCN(dropout_rate=DROPOUT),
}

test_accs   = {}
best_states = {}   # stores best weights for every model — used by the weight-saving cell

for name, build_fn in model_registry.items():
    print(f"\n{'='*60}")
    print(f"  Training: {name}")
    print(f"{'='*60}")

    model     = build_fn().to(device)
    optimizer = optim.Adam(model.parameters(), lr=LR)
    scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.5,
                                  patience=5, min_lr=1e-6)
    criterion = nn.CrossEntropyLoss()

    best_val_acc = float('-inf')
    best_state   = None
    bad_epochs   = 0

    for epoch in range(MAX_EPOCHS):
        tr_loss  = train(model, train_loader, criterion, optimizer, device)
        val_acc  = evaluate(model, dev_loader, device)
        curr_lr  = optimizer.param_groups[0]['lr']

        # Best-state checkpoint
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state   = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        # Learning-rate scheduler (monitors dev accuracy)
        scheduler.step(val_acc)

        # Early stopping
        if val_acc >= (best_val_acc - MIN_DELTA):
            bad_epochs = 0
        else:
            bad_epochs += 1

        print(f"  Epoch {epoch+1:3d}: loss={tr_loss:.4f}  dev={val_acc*100:.2f}%  "
              f"best={best_val_acc*100:.2f}%  lr={curr_lr:.2e}")

        if bad_epochs >= PATIENCE:
            print(f"  Early stopping at epoch {epoch+1}.")
            break

    # Restore best weights
    model.load_state_dict(best_state)
    best_states[name] = best_state   # keep a copy for weight saving

    print(f"\n--- {name} — Test Set Evaluation ---")
    test_acc = evaluateFinal(model, test_loader, device)
    test_accs[name] = test_acc

---
## Results Summary

In [19]:
print(f"\n{'Model':<18} {'Test Acc':>10}")
print("-" * 30)
for name in model_registry:
    acc = test_accs.get(name, float('nan')) * 100
    print(f"{name:<18} {acc:>9.2f}%")

names = list(model_registry.keys())
accs  = [test_accs.get(n, 0) * 100 for n in names]


Model                Test Acc
------------------------------
EEGNet                 86.43%
ShallowConvNet         81.07%
DeepConvNet            59.64%
CNN_LSTM               51.43%
EMG_TCN                93.57%


---
## Save Best Model Weights

Identifies the best-performing model by test accuracy and saves its weights
to `weights/`. The checkpoint file is read by `4_model_retrain_full.ipynb`.

In [20]:
import os

WEIGHTS_DIR = 'weights'
os.makedirs(WEIGHTS_DIR, exist_ok=True)

# ── Pick the best model by test accuracy ──────────────────────────────────────
best_model_name = max(test_accs, key=test_accs.get)
best_model_acc  = test_accs[best_model_name]

print(f"Best model : {best_model_name}")
print(f"Test acc   : {best_model_acc * 100:.2f}%")

# ── Save checkpoint ───────────────────────────────────────────────────────────
save_path = os.path.join(WEIGHTS_DIR, f"{best_model_name}_best.pt")
torch.save({
    'model_name':  best_model_name,
    'test_acc':    best_model_acc,
    'dropout':     DROPOUT,
    'state_dict':  best_states[best_model_name],
}, save_path)

print(f"Saved → {save_path}")

Best model : EMG_TCN
Test acc   : 93.57%
Saved → weights/EMG_TCN_best.pt
